# CS5812 - KNN and 3-layer ANN on the Smart Meters in London datasetAuthor: Rahim. Individual modelling notebook accompanying the report.Sections in this notebook follow the report:1. Load + train/test split2. KNN (k tuning, final fit, metrics)3. 3-layer ANN (64-32-16) + permutation feature importance4. Comparison and 5-fold cross-validation5. Leave-one-feature-group-out ablation

## 1. Load and split

In [ ]:
import numpy as npimport pandas as pdfrom sklearn.preprocessing import StandardScalerseed = 50np.random.seed(seed)df = pd.read_csv("daily_sample.csv")df = pd.get_dummies(df, columns=["Acorn_grouped", "stdorToU"], drop_first=True)df["log_energy_sum"] = np.log1p(df["energy_sum"])target = "energy_sum"features = [c for c in df.columns            if c not in ["LCLid", "date", target, "log_energy_sum"]]dates  = sorted(df["date"].unique())cutoff = dates[int(0.8 * len(dates))]train  = df[df["date"] <  cutoff]test   = df[df["date"] >= cutoff]X_train = train[features].values.astype(float)X_test  = test [features].values.astype(float)y_train = train[target].valuesy_test  = test [target].valuesy_train_log = train["log_energy_sum"].valuessc = StandardScaler()X_train = sc.fit_transform(X_train)X_test  = sc.transform(X_test)print("Train:", X_train.shape, "Test:", X_test.shape)

## 2. KNN (machine-learning model)

In [ ]:
from sklearn.neighbors import KNeighborsRegressorfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score# Tune kks = [3, 5, 10, 20, 50]scores = {}for k in ks:    m = KNeighborsRegressor(n_neighbors=k, weights="distance")    m.fit(X_train, y_train)    p = m.predict(X_test)    scores[k] = np.sqrt(mean_squared_error(y_test, p))    print(f"k={k:>3}  RMSE={scores[k]:.3f}")best_k = min(scores, key=scores.get)print("\nChosen k =", best_k)# Refit at chosen kknn = KNeighborsRegressor(n_neighbors=best_k, weights="distance")knn.fit(X_train, y_train)y_pred_knn = knn.predict(X_test)mae_knn  = mean_absolute_error(y_test, y_pred_knn)rmse_knn = np.sqrt(mean_squared_error(y_test, y_pred_knn))r2_knn   = r2_score(y_test, y_pred_knn)print(f"\nKNN  MAE={mae_knn:.3f}  RMSE={rmse_knn:.3f}  R2={r2_knn:.3f}")

## 3. 3-layer ANN (64-32-16, deep-learning model)

In [ ]:
import tensorflow as tffrom tensorflow.keras import layers, models, callbackstf.random.set_seed(seed)ann = models.Sequential([    layers.Input(shape=(X_train.shape[1],)),    layers.Dense(64, activation="relu"),    layers.Dense(32, activation="relu"),    layers.Dense(16, activation="relu"),    layers.Dense(1),                          # linear output])ann.compile(optimizer="adam", loss="mse", metrics=["mae"])stop = callbacks.EarlyStopping(patience=5, restore_best_weights=True,                               monitor="val_loss")ann.fit(X_train, y_train_log,        validation_split=0.15,        epochs=50, batch_size=256,        callbacks=[stop], verbose=0)y_pred_ann = np.expm1(ann.predict(X_test, verbose=0).flatten())mae_ann  = mean_absolute_error(y_test, y_pred_ann)rmse_ann = np.sqrt(mean_squared_error(y_test, y_pred_ann))r2_ann   = r2_score(y_test, y_pred_ann)print(f"ANN (64-32-16)  MAE={mae_ann:.3f}  RMSE={rmse_ann:.3f}  R2={r2_ann:.3f}")

### 3.1  ANN feature importance (permutation method)

In [ ]:
import matplotlib.pyplot as pltbase_pred = np.expm1(ann.predict(X_test, verbose=0).flatten())base_mse  = mean_squared_error(y_test, base_pred)imp = {}for i, name in enumerate(features):    Xp = X_test.copy()    np.random.shuffle(Xp[:, i])    p = np.expm1(ann.predict(Xp, verbose=0).flatten())    imp[name] = mean_squared_error(y_test, p) - base_mseann_importance = (pd.Series(imp).clip(lower=0)                  .sort_values(ascending=False))print(ann_importance.head(15))share = ann_importance / ann_importance.sum()share.head(15)[::-1].plot(kind="barh", figsize=(7, 5), color="tab:orange")plt.xlabel("Relative importance")plt.title("ANN feature importance (permutation)")plt.tight_layout()plt.show()

## 4. Comparison and 5-fold cross-validation

In [ ]:
from sklearn.model_selection import KFold# Headline summarysummary = pd.DataFrame(    [(f"KNN (k={best_k})",  mae_knn, rmse_knn, r2_knn),     ("ANN (64-32-16)",     mae_ann, rmse_ann, r2_ann)],    columns=["Model", "MAE", "RMSE", "R2"]).round(3)print(summary.to_string(index=False))# 5-fold CV on a 50k subsample (KNN is slow on the full set)rng = np.random.default_rng(seed)sub = rng.choice(len(X_train), size=50_000, replace=False)Xs, ys = X_train[sub], y_train[sub]kf = KFold(n_splits=5, shuffle=True, random_state=seed)cv_knn, cv_ann = [], []for tr, va in kf.split(Xs):    m = KNeighborsRegressor(n_neighbors=best_k, weights="distance").fit(Xs[tr], ys[tr])    cv_knn.append(np.sqrt(mean_squared_error(ys[va], m.predict(Xs[va]))))    nn = models.Sequential([        layers.Input(shape=(Xs.shape[1],)),        layers.Dense(64, activation="relu"),        layers.Dense(32, activation="relu"),        layers.Dense(16, activation="relu"),        layers.Dense(1),    ])    nn.compile(optimizer="adam", loss="mse")    nn.fit(Xs[tr], np.log1p(ys[tr]),           epochs=15, batch_size=256, verbose=0)    p = np.expm1(nn.predict(Xs[va], verbose=0).flatten())    cv_ann.append(np.sqrt(mean_squared_error(ys[va], p)))print(f"\nKNN 5-fold RMSE = {np.mean(cv_knn):.3f} +/- {np.std(cv_knn):.3f}")print(f"ANN 5-fold RMSE = {np.mean(cv_ann):.3f} +/- {np.std(cv_ann):.3f}")# Holdout vs CV bar chartxs_pos = np.arange(2)plt.figure(figsize=(7, 4))plt.bar(xs_pos - 0.2, summary["RMSE"], width=0.4, label="Holdout test set")plt.bar(xs_pos + 0.2,        [np.mean(cv_knn), np.mean(cv_ann)],        yerr=[np.std(cv_knn), np.std(cv_ann)],        width=0.4, capsize=5, label="5-fold CV (mean +/- SD)")plt.xticks(xs_pos, summary["Model"])plt.ylabel("RMSE (kWh)")plt.title("Holdout vs 5-fold cross-validation")plt.legend(); plt.tight_layout(); plt.show()

## 5. Leave-one-feature-group-out ablation

In [ ]:
groups = {    "lag":      ["energy_lag1", "energy_lag7"],    "weather":  ["temperatureMax","temperatureMin","temperatureHigh",                 "temperatureLow","humidity","windSpeed",                 "pressure","visibility"],    "calendar": ["day_of_week","is_weekend","month","season","is_holiday"],    "context":  [c for c in features                 if c.startswith(("Acorn_grouped_","stdorToU_"))],}base_rmse = np.sqrt(mean_squared_error(y_test, y_pred_knn))print(f"Full features  RMSE = {base_rmse:.3f}")for name, cols in groups.items():    keep = [i for i, f in enumerate(features) if f not in cols]    m = KNeighborsRegressor(n_neighbors=best_k, weights="distance")    m.fit(X_train[:, keep], y_train)    p = m.predict(X_test[:, keep])    rmse = np.sqrt(mean_squared_error(y_test, p))    print(f"no {name:8s}   RMSE = {rmse:.3f}   delta = {rmse-base_rmse:+.3f}")